# Verb Noise Control Calibration

This notebook is a loader and plotting view for saved per-model Gaussian verb-noise calibration artifacts. Run the expensive sweep on GPU first, then load all four models together here.

Suggested workflow:

1. Submit one job per model:
   `python animacy-circuit/scripts/executable/submit_verb_noise_control_calibration_jobs.py`
2. Wait for the four jobs to finish.
3. Set `OUTPUT_DAY` below if needed, then run the notebook to load and compare all four models in one figure.
4. If a model still selects the maximum tested sigma multiplier, treat that as a calibration ceiling hit and rerun calibration before trusting the downstream control evaluation.

In [1]:
from pathlib import Path
import json
import sys

import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

cwd = Path.cwd().resolve()
executable_dir = None
for base in (cwd, *cwd.parents):
    for candidate in (
        base / "scripts" / "executable",
        base / "animacy-circuit" / "scripts" / "executable",
    ):
        if candidate.is_dir():
            executable_dir = candidate
            break
    if executable_dir is not None:
        break
if executable_dir is None:
    raise FileNotFoundError("Could not locate scripts/executable for notebook imports.")
if str(executable_dir) not in sys.path:
    sys.path.insert(0, str(executable_dir))

from circuit_finder_core import (
    DEFAULT_COMMON_TOKENIZATION_FILTER_MODELS,
    canonical_model_name,
    resolve_animacy_circuit_root,
    safe_model_name,
)
from control_runners import DEFAULT_VERB_NOISE_SIGMA_MULTIPLIERS


In [2]:
MODEL_NAMES = tuple(DEFAULT_COMMON_TOKENIZATION_FILTER_MODELS)
# OUTPUT_DAY = "2026-06-05_full_h100"
OUTPUT_DAY = "2026-06-06_recalibration_h100"
CONTROL_NAME = "verb_noise_calibration"
EXPECTED_SIGMA_MULTIPLIERS = tuple(float(value) for value in DEFAULT_VERB_NOISE_SIGMA_MULTIPLIERS)

assert len(MODEL_NAMES) == 4, f"Expected 4 models, got {len(MODEL_NAMES)}"


In [3]:
project_root = resolve_animacy_circuit_root(Path.cwd())


def calibration_dir(day: str, model_name: str) -> Path:
    return (
        project_root
        / "results"
        / day
        / safe_model_name(canonical_model_name(model_name))
        / "controls"
        / CONTROL_NAME
    )


def calibration_paths(day: str, model_name: str) -> dict[str, Path]:
    base_dir = calibration_dir(day, model_name)
    return {
        "base_dir": base_dir,
        "summary": base_dir / "calibration_summary.json",
        "selected": base_dir / "selected_sigma.json",
        "sweep": base_dir / "sigma_sweep.csv",
    }


def available_days() -> list[str]:
    results_root = project_root / "results"
    if not results_root.is_dir():
        return []
    days = []
    for candidate in sorted(results_root.iterdir()):
        if not candidate.is_dir():
            continue
        if all(calibration_paths(candidate.name, model_name)["summary"].is_file() for model_name in MODEL_NAMES):
            days.append(candidate.name)
    return days


def resolve_output_day(requested_day: str | None) -> str:
    if requested_day is not None:
        missing = [
            canonical_model_name(model_name)
            for model_name in MODEL_NAMES
            if not calibration_paths(requested_day, model_name)["summary"].is_file()
        ]
        if missing:
            raise FileNotFoundError(
                f"Missing calibration summaries for day {requested_day}: {missing}"
            )
        return requested_day
    days = available_days()
    if not days:
        raise FileNotFoundError(
            "No complete four-model calibration day was found. Submit jobs first or set OUTPUT_DAY explicitly."
        )
    return days[-1]


resolved_day = resolve_output_day(OUTPUT_DAY)
print(f"Loading calibration artifacts for day {resolved_day}")


Loading calibration artifacts for day 2026-06-06_recalibration_h100


In [4]:
model_results = {}
summary_rows = []

for model_name in MODEL_NAMES:
    resolved_model_name = canonical_model_name(model_name)
    paths = calibration_paths(resolved_day, resolved_model_name)
    summary = json.loads(paths["summary"].read_text(encoding="utf-8"))
    selected = json.loads(paths["selected"].read_text(encoding="utf-8"))
    sweep_df = pd.read_csv(paths["sweep"]).sort_values("sigma_multiplier").reset_index(drop=True)
    if "absolute_mean_margin" not in sweep_df.columns:
        sweep_df["absolute_mean_margin"] = sweep_df["margin_mean"].abs()
    if "mean_absolute_margin" not in sweep_df.columns and "absolute_margin_mean" in sweep_df.columns:
        sweep_df["mean_absolute_margin"] = sweep_df["absolute_margin_mean"]
    selected_absolute_mean_margin = float(selected.get("absolute_mean_margin", abs(selected["margin_mean"])))
    sweep_max_multiplier = float(sweep_df["sigma_multiplier"].max())
    selected_at_sweep_ceiling = abs(float(selected["sigma_multiplier"]) - sweep_max_multiplier) < 1e-12

    item = {
        **summary,
        "selected": selected,
        "sweep_df": sweep_df,
        "paths": {
            **summary.get("paths", {}),
            "summary_json": str(paths["summary"]),
            "selected_sigma_json": str(paths["selected"]),
            "sigma_sweep_csv": str(paths["sweep"]),
        },
    }
    model_results[resolved_model_name] = item
    summary_rows.append(
        {
            "model_name": resolved_model_name,
            "target_filtered_count": int(item["target_filtered_count"]),
            "validated_control_rows": int(item["validated_control_rows"]),
            "activation_scale": float(item["activation_scale"]),
            "original_margin_mean": float(item["original_summary"]["margin_mean"]),
            "selected_sigma_multiplier": float(selected["sigma_multiplier"]),
            "selected_sigma": float(selected["sigma"]),
            "selected_margin_mean": float(selected["margin_mean"]),
            "selected_absolute_mean_margin": selected_absolute_mean_margin,
            "selected_mean_absolute_margin": float(selected.get("mean_absolute_margin", selected.get("absolute_margin_mean", float("nan")))),
            "warning_overcorruption": bool(selected["warning_overcorruption"]),
            "sweep_max_multiplier": sweep_max_multiplier,
            "selected_at_sweep_ceiling": selected_at_sweep_ceiling,
            "sweep_matches_current_default": tuple(float(value) for value in summary.get("sigma_multipliers", [])) == EXPECTED_SIGMA_MULTIPLIERS,
            "selected_sigma_json": str(paths["selected"]),
        }
    )

summary_df = pd.DataFrame(summary_rows).sort_values("model_name").reset_index(drop=True)
summary_df


,model_name,target_filtered_count,validated_control_rows,activation_scale,original_margin_mean,selected_sigma_multiplier,selected_sigma,selected_margin_mean,selected_absolute_mean_margin,selected_mean_absolute_margin,warning_overcorruption,sweep_max_multiplier,selected_at_sweep_ceiling,sweep_matches_current_default,selected_sigma_json
0,Qwen/Qwen3-4B,4300,4300,0.021891,2.150185,8.0,0.175125,0.061781,0.061781,0.097293,False,8.0,True,True,/gpfs/home4/spunzo/grammatical-circuits/animac...
1,google/gemma-3-4b-pt,4371,4371,0.972509,3.241368,8.0,7.780073,0.043750,0.043750,0.157222,False,8.0,True,True,/gpfs/home4/spunzo/grammatical-circuits/animac...
2,gpt2,4721,4721,0.198607,2.585372,8.0,1.588857,0.053431,0.053431,0.093184,False,8.0,True,True,/gpfs/home4/spunzo/grammatical-circuits/animac...
3,meta-llama/Llama-3.2-3B,4734,4734,0.020813,2.313783,8.0,0.166505,0.041718,0.041718,0.122210,False,8.0,True,True,/gpfs/home4/spunzo/grammatical-circuits/animac...


In [5]:
subplot_titles = [canonical_model_name(name) for name in MODEL_NAMES]
fig = make_subplots(
    rows=2,
    cols=2,
    subplot_titles=subplot_titles,
    horizontal_spacing=0.10,
    vertical_spacing=0.16,
)

for idx, model_name in enumerate(subplot_titles):
    row = idx // 2 + 1
    col = idx % 2 + 1
    item = model_results[model_name]
    sweep_df = item["sweep_df"].sort_values("sigma_multiplier").reset_index(drop=True)
    selected = item["selected"]
    original_abs_margin = float(item["original_absolute_margin_mean"])
    sweep_max_multiplier = float(sweep_df["sigma_multiplier"].max())
    selected_at_sweep_ceiling = abs(float(selected["sigma_multiplier"]) - sweep_max_multiplier) < 1e-12

    fig.add_trace(
        go.Scatter(
            x=sweep_df["sigma_multiplier"],
            y=sweep_df["margin_mean"],
            mode="lines+markers",
            name="margin_mean",
            legendgroup="margin_mean",
            showlegend=(idx == 0),
            line=dict(color="#1f77b4"),
        ),
        row=row,
        col=col,
    )
    fig.add_trace(
        go.Scatter(
            x=sweep_df["sigma_multiplier"],
            y=sweep_df["absolute_mean_margin"],
            mode="lines+markers",
            name="absolute_mean_margin",
            legendgroup="absolute_mean_margin",
            showlegend=(idx == 0),
            line=dict(color="#d62728"),
        ),
        row=row,
        col=col,
    )
    fig.add_trace(
        go.Scatter(
            x=[selected["sigma_multiplier"]],
            y=[float(selected.get("absolute_mean_margin", abs(selected["margin_mean"])))],
            mode="markers",
            name="selected_sigma",
            legendgroup="selected_sigma",
            showlegend=(idx == 0),
            marker=dict(color="#111111", size=10, symbol="diamond"),
            customdata=[[selected["sigma"], selected["margin_mean"]]],
            hovertemplate="selected multiplier=%{x}<br>selected sigma=%{customdata[0]:.4g}<br>abs margin=%{y:.4g}<br>margin=%{customdata[1]:.4g}<extra></extra>",
        ),
        row=row,
        col=col,
    )

    fig.add_vline(
        x=sweep_max_multiplier,
        line_dash="dash",
        line_color="#555555",
        opacity=0.5,
        row=row,
        col=col,
    )

    for fraction, color in ((0.10, "gray"), (0.20, "orange"), (0.25, "red")):
        fig.add_hline(
            y=original_abs_margin * fraction,
            line_dash="dot",
            line_color=color,
            opacity=0.6,
            row=row,
            col=col,
        )

    if selected_at_sweep_ceiling:
        fig.add_annotation(
            x=sweep_max_multiplier,
            y=float(selected.get("absolute_mean_margin", abs(selected["margin_mean"]))),
            text="ceiling hit",
            showarrow=True,
            arrowhead=2,
            ax=-45,
            ay=-35,
            font=dict(color="#b22222"),
            row=row,
            col=col,
        )

    fig.update_xaxes(title_text="sigma multiplier", row=row, col=col)
    fig.update_yaxes(title_text="metric", row=row, col=col)

fig.update_layout(
    title=f"Verb-noise sigma sweep across all four models ({resolved_day})",
    height=900,
    width=1200,
)
fig.show()


In [6]:
pd.DataFrame(
    [
        {
            "model_name": model_name,
            "selected_sigma": float(item["selected"]["sigma"]),
            "selected_sigma_multiplier": float(item["selected"]["sigma_multiplier"]),
            "selected_at_sweep_ceiling": bool(
                abs(
                    float(item["selected"]["sigma_multiplier"])
                    - float(item["sweep_df"]["sigma_multiplier"].max())
                )
                < 1e-12
            ),
            "summary_json": item["paths"]["summary_json"],
            "selected_sigma_json": item["paths"]["selected_sigma_json"],
            "sigma_sweep_csv": item["paths"]["sigma_sweep_csv"],
            "control_eval_stub": (
                f"python animacy-circuit/scripts/executable/run.py control-verb-noise "
                f"--model {model_name} --dataset-filter-model gpt2 "
                f"--target-filter-policy model_success --sigma {float(item['selected']['sigma']):.12g} "
                f"--main-experiment-path <full_model_artifact_or_dir> --output-day <new_day>"
            ),
        }
        for model_name, item in model_results.items()
    ]
).sort_values("model_name").reset_index(drop=True)


,model_name,selected_sigma,selected_sigma_multiplier,selected_at_sweep_ceiling,summary_json,selected_sigma_json,sigma_sweep_csv,control_eval_stub
0,Qwen/Qwen3-4B,0.175125,8.0,True,/gpfs/home4/spunzo/grammatical-circuits/animac...,/gpfs/home4/spunzo/grammatical-circuits/animac...,/gpfs/home4/spunzo/grammatical-circuits/animac...,python animacy-circuit/scripts/executable/run....
1,google/gemma-3-4b-pt,7.780073,8.0,True,/gpfs/home4/spunzo/grammatical-circuits/animac...,/gpfs/home4/spunzo/grammatical-circuits/animac...,/gpfs/home4/spunzo/grammatical-circuits/animac...,python animacy-circuit/scripts/executable/run....
2,gpt2,1.588857,8.0,True,/gpfs/home4/spunzo/grammatical-circuits/animac...,/gpfs/home4/spunzo/grammatical-circuits/animac...,/gpfs/home4/spunzo/grammatical-circuits/animac...,python animacy-circuit/scripts/executable/run....
3,meta-llama/Llama-3.2-3B,0.166505,8.0,True,/gpfs/home4/spunzo/grammatical-circuits/animac...,/gpfs/home4/spunzo/grammatical-circuits/animac...,/gpfs/home4/spunzo/grammatical-circuits/animac...,python animacy-circuit/scripts/executable/run....
